# L5: Automatize o planejamento de eventos
* Nesta lição, você aprenderá mais sobre Tarefas. 

In [36]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [37]:
from crewai import Agent, Task, Crew

In [38]:
import os
from utils import get_openai_api_key, pretty_print_result
from utils import get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

In [39]:
os.environ["OPENAI_MODEL_NAME"] = "gpt-3.5-turbo"

crewAI Tools

In [40]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool
from langchain.utilities import SerpAPIWrapper
# Initialize the tools
search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

# Creating Agents

In [41]:
# Agente 1: Coordenador de Local
venue_coordinator = Agent(
    role="Coordenador de Local",
    goal="Identificar e reservar um local apropriado "
         "com base nos requisitos do evento",
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "Com um senso aguçado de espaço e "
        "compreensão da logística de eventos, "
        "você se destaca em encontrar e garantir "
        "o local perfeito que atenda ao tema do evento, "
        "ao tamanho e às restrições de orçamento."
    )
)


LLM value is None


In [42]:
# Agente 2: Gerente de Logística
logistics_manager = Agent(
    role="Gerente de Logística",
    goal=(
        "Gerenciar toda a logística do evento, "
        "incluindo catering e equipamentos"
    ),
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "Organizado e orientado para os detalhes, "
        "você garante que cada aspecto logístico do evento, "
        "desde o catering até a instalação de equipamentos, "
        "seja executado de forma impecável para criar uma experiência perfeita."
    )
)


LLM value is None


In [43]:
# Agente 3: Agente de Marketing e Comunicação
marketing_communications_agent = Agent(
    role="Agente de Marketing e Comunicação",
    goal="Promover o evento de forma eficaz e "
         "comunicar-se com os participantes",
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "Criativo e comunicativo, "
        "você cria mensagens envolventes e "
        "interage com potenciais participantes "
        "para maximizar a exposição e a participação no evento."
    )
)

LLM value is None


# Criando Objeto Pydantic de Local
* Crie uma classe VenueDetails usando Pydantic BaseModel.
* Os agentes preencherão este objeto com informações sobre diferentes locais, criando diferentes instâncias dele.

In [44]:
from pydantic import BaseModel

# Define um modelo Pydantic para os detalhes do local
# (demonstrando saída como Pydantic)
class DetalhesLocal(BaseModel):
    nome: str
    endereco: str
    capacidade: int
    status_reserva: str

# Creating Tasks

* Usando output_json, você pode especificar a estrutura da saída desejada.
* Usando output_file, você pode obter sua saída em um arquivo.
* Ao definir human_input=True, a tarefa solicitará feedback humano (goste você dos resultados ou não) antes de finalizá-la.

In [45]:
venue_task = Task(
    description="Encontre um local em {event_city} "
                "que atenda aos critérios para {event_topic}.",
    expected_output="Todos os detalhes de um local especificamente escolhido "
                    "para acomodar o evento.",
    human_input=True,
    output_json=DetalhesLocal,
    output_file="detalhes_local.json",  
      # Salva os detalhes do local como um arquivo JSON
    agent=venue_coordinator
)

* Ao definir async_execution=True, significa que a tarefa pode ser executada em paralelo com as tarefas que vêm depois dela.

In [46]:
logistics_task = Task(
    description="Coordene o catering e "
                "os equipamentos para um evento "
                "com {expected_participants} participantes "
                "em {tentative_date}.",
    expected_output="Confirmação de todos os arranjos logísticos, "
                    "incluindo catering e instalação de equipamentos.",
    human_input=True,
    agent=logistics_manager
)

In [47]:
marketing_task = Task(
    description="Promova o {event_topic} "
                "com o objetivo de engajar pelo menos "
                "{expected_participants} potenciais participantes.",
    expected_output="Relatório sobre as atividades de marketing "
                    "e o engajamento dos participantes, formatado em markdown.",
    async_execution=True,
    output_file="relatorio_marketing.md",  # Salva o relatório como um arquivo de texto
    agent=marketing_communications_agent
)

# Creating the Crew

* Observação: como você definiu async_execution=True para tarefas Logistics_task e marketing_task, agora a ordem delas não importa na lista de tarefas.

In [48]:
# Define the crew with agents and tasks
event_management_crew = Crew(
    agents=[venue_coordinator, 
            logistics_manager, 
            marketing_communications_agent],
    
    tasks=[venue_task, 
           logistics_task, 
           marketing_task],
    
    verbose=True
)

2025-01-27 17:55:19,733 - 27084 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


# Running the Crew
* Atribua as entradas para a execução da Crew

In [49]:
event_details = {
    'event_topic': "Conferência de Inovação Tecnológica",
    'event_description': "Um encontro de inovadores em tecnologia "
                         "e líderes da indústria "
                         "para explorar as tecnologias do futuro.",
    'event_city': "São Francisco",
    'tentative_date': "2024-09-15",
    'expected_participants': 500,
    'budget': 20000,
    'venue_type': "Salão de Conferências"
}

Observação:
* Como você definiu human_input=True para algumas tarefas, a execução solicitará sua entrada antes de terminar a execução.
* Quando solicitar feedback, use o ponteiro do mouse para clicar primeiro na caixa de texto antes de digitar qualquer coisa.

In [ ]:
result = event_management_crew.kickoff(inputs=event_details)

# Agent: Coordenador de Local
## Task: Encontre um local em São Francisco que atenda aos critérios para Conferência de Inovação Tecnológica.


# Agent: Coordenador de Local
## Using tool: Search the internet with Serper
## Tool Input: 
"{\"search_query\": \"Technology Innovation Conference venue in San Francisco\"}"
## Tool Output: 
{'searchParameters': {'q': 'Technology Innovation Conference venue in San Francisco', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': 'Top 12 tech conferences in San Francisco for 2024 - TravelPerk', 'link': 'https://www.travelperk.com/blog/top-tech-conferences-in-san-francisco/', 'snippet': 'Santa Clara Convention Center, Santa Clara ... With over 200 speakers, the San Francisco Tech Summit offers an action-packed agenda for ...', 'position': 1}, {'title': 'Event Details San Francisco - Tech Summit', 'link': 'https://techsummit.tech/san-francisco-tech-conference/?srsltid=AfmBOorw2QVLuUg6z1vl18nvlqV_g_gxLNKXYTE10NX3iOsytyTCrkXQ', 'sn

* Exiba o arquivo local_details.json gerado.

In [ ]:
import json
from pprint import pprint

with open('venue_details.json') as f:
   data = json.load(f)

pprint(data)



> Entering new CrewAgentExecutor chain...


> Entering new CrewAgentExecutor chain...
Preciso garantir que todos os arranjos logísticos, incluindo catering e instalação de equipamentos, estejam perfeitamente organizados para o evento com 500 participantes em 15 de setembro de 2024 em São Francisco.

Action: Read website content
Action Input: {"website_url": "https://www.cateringsf.com"}Preciso pensar em estratégias criativas para promover a Conferência de Inovação Tecnológica e engajar potenciais participantes.

Action: Search the internet
Action Input: {"search_query": "Ideias criativas para promover eventos e engajar participantes"} 



 

{'message': 'Unauthorized.', 'statusCode': 403}

Action: Read website content
Action Input: {"website_url": "www.eventbrite.com/blog/creative-ideas-promote-engage-event/"} 

I encountered an error while trying to use the tool. This was the error: Invalid URL 'www.eventbrite.com/blog/creative-ideas-promote-engage-event/': No scheme supplied. Perha

{'capacidade': 200,
 'endereco': '123 Main Street, São Francisco',
 'nome': 'Tech Hub SF',
 'status_reserva': 'Disponível'}


* Exiba o arquivo marketing_report.md gerado.

Observação: após a execução bem-sucedida da execução inicial, aguarde mais 45 segundos para que o arquivo marketing_report.md seja gerado. Se você tentar executar o código abaixo antes de o arquivo ser gerado, sua saída será semelhante a:

  - relatório_marketing.md

Se você vir esta saída, espere mais um pouco e tente novamente.

In [ ]:
from IPython.display import Markdown
Markdown("marketing_report.md")

marketing_report.md